# Multi-Source Consolidation with Complementary Registers — Public Demo (2023)

This notebook is a **sanitized portfolio version** of a more advanced operational consolidation workflow.

Compared with the 2024 workflow, this version includes both:

- same-structure update workbooks
- complementary administrative/status registers

All data in this notebook is synthetic.  
Real organization names, identifiers, staff names, internal document numbers, file paths, and source files are intentionally excluded.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Synthetic master dataset

In [ ]:
master = pd.DataFrame({
    "RECORD_ID": [f"REC-{i:03d}" for i in range(1, 9)],
    "AGREEMENT_ID": [f"AGR-{i:03d}" for i in range(1, 9)],
    "REGION": ["NORTH", "SOUTH", "CENTRAL", "NORTH", "SOUTH", "CENTRAL", "NORTH", "SOUTH"],
    "PROCESS_STATUS": ["DONE", "DONE", "PENDING", "DONE", "PENDING", "DONE", "DONE", "PENDING"],
    "CLOSURE_STATUS": ["PENDING", "PENDING", "PENDING", "DONE", "PENDING", "DONE", "PENDING", "PENDING"],
    "ADMIN_STATUS": [None, None, None, "SENT", None, "SENT", None, None],
    "ADMIN_REFERENCE": [None, None, None, "REF-004", None, "REF-006", None, None],
    "RESOLUTION_NO": [None, None, None, None, None, "RES-006", None, None],
    "RESOLUTION_DATE": [None, None, None, None, None, "2026-06-15", None, None]
})

master

## 2. Same-structure update sources

In [ ]:
update_1 = master.copy()
update_1.loc[update_1["RECORD_ID"] == "REC-003", "PROCESS_STATUS"] = "DONE"
update_1.loc[update_1["RECORD_ID"] == "REC-007", "CLOSURE_STATUS"] = "DONE"

update_2 = master.copy()
update_2.loc[update_2["RECORD_ID"] == "REC-005", "CLOSURE_STATUS"] = "DONE"

update_sources = {
    "FOLLOW_UP_1": update_1,
    "FOLLOW_UP_2": update_2
}

## 3. Complementary administrative and resolution registers

These sources do not replace the master dataset.  
They provide additional fields or more authoritative values for specific columns.

In [ ]:
admin_register = pd.DataFrame({
    "AGREEMENT_ID": ["AGR-001", "AGR-002", "AGR-003", "AGR-005", "AGR-007"],
    "ADMIN_STATUS": ["PENDING", "SENT", "PENDING", "SENT", "IN PROCESS"],
    "ADMIN_REFERENCE": [None, "REF-002-EXT", None, "REF-005", None],
    "ADMIN_OBSERVATION": [None, None, "REVIEW REQUIRED", None, None]
})

resolution_register = pd.DataFrame({
    "AGREEMENT_ID": ["AGR-004", "AGR-006", "AGR-008"],
    "RESOLUTION_NO": ["RES-004", "RES-006-NEW", "RES-008"],
    "RESOLUTION_DATE": ["2026-05-10", "2026-06-20", "2026-07-02"]
})

## 4. Clean and validate identifiers

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return None
    value = " ".join(str(value).replace("\n", " ").replace("\r", " ").split()).strip()
    return value if value else None

for col in ["RECORD_ID", "AGREEMENT_ID"]:
    master[col] = master[col].map(clean_text)

print("Rows:", len(master))
print("Unique record IDs:", master["RECORD_ID"].nunique())
print("Unique agreement IDs:", master["AGREEMENT_ID"].nunique())
print("Duplicate record IDs:", master["RECORD_ID"].duplicated().sum())

## 5. Detect differences from same-structure update sources

In [ ]:
master_idx = master.set_index("RECORD_ID")
update_idx = {
    name: df.set_index("RECORD_ID")
    for name, df in update_sources.items()
}

differences = []

for source_name, df_source in update_idx.items():
    for record_id in master_idx.index:
        for column in master_idx.columns:
            base_value = clean_text(master_idx.at[record_id, column])
            source_value = clean_text(df_source.at[record_id, column])

            if base_value != source_value:
                differences.append({
                    "SOURCE": source_name,
                    "RECORD_ID": record_id,
                    "COLUMN": column,
                    "BASE_VALUE": base_value,
                    "SOURCE_VALUE": source_value
                })

differences_df = pd.DataFrame(differences)
differences_df

## 6. Separate unique changes from conflicts

In [ ]:
grouped = (
    differences_df
    .groupby(["RECORD_ID", "COLUMN"])["SOURCE_VALUE"]
    .agg(lambda s: list(pd.unique(s)))
    .reset_index(name="PROPOSED_VALUES")
)

grouped["VALUE_COUNT"] = grouped["PROPOSED_VALUES"].apply(len)

unique_updates = grouped[grouped["VALUE_COUNT"] == 1].copy()
conflicts = grouped[grouped["VALUE_COUNT"] > 1].copy()

print("Unique updates:", len(unique_updates))
print("Conflicts:", len(conflicts))

## 7. Apply non-conflicting updates

In [ ]:
consolidated = master_idx.copy()
traceability = []

for _, row in unique_updates.iterrows():
    record_id = row["RECORD_ID"]
    column = row["COLUMN"]
    new_value = row["PROPOSED_VALUES"][0]

    old_value = consolidated.at[record_id, column]
    consolidated.at[record_id, column] = new_value

    traceability.append({
        "SOURCE": "FOLLOW_UP",
        "RECORD_ID": record_id,
        "COLUMN": column,
        "OLD_VALUE": old_value,
        "NEW_VALUE": new_value,
        "RULE": "UNIQUE_NON_CONFLICTING_UPDATE"
    })

consolidated = consolidated.reset_index()

## 8. Integrate the administrative register using field-level priority rules

Example public rules:

- `ADMIN_STATUS`: complementary register takes priority
- `ADMIN_REFERENCE`: complementary register fills blanks only
- `ADMIN_OBSERVATION`: complementary register takes priority when present

In [ ]:
consolidated = consolidated.merge(
    admin_register,
    on="AGREEMENT_ID",
    how="left",
    suffixes=("", "_ADMIN")
)

# ADMIN_STATUS: complementary source prevails when available
mask = consolidated["ADMIN_STATUS_ADMIN"].notna()
consolidated.loc[mask, "ADMIN_STATUS"] = consolidated.loc[mask, "ADMIN_STATUS_ADMIN"]

# ADMIN_REFERENCE: only fill blanks
mask = (
    consolidated["ADMIN_REFERENCE"].isna()
    & consolidated["ADMIN_REFERENCE_ADMIN"].notna()
)
consolidated.loc[mask, "ADMIN_REFERENCE"] = consolidated.loc[mask, "ADMIN_REFERENCE_ADMIN"]

# Optional complementary field
consolidated["ADMIN_OBSERVATION_FINAL"] = consolidated["ADMIN_OBSERVATION"]

consolidated = consolidated.drop(
    columns=["ADMIN_STATUS_ADMIN", "ADMIN_REFERENCE_ADMIN", "ADMIN_OBSERVATION"]
)

## 9. Integrate the resolution register

The dedicated resolution source is treated as authoritative for resolution number and date.

In [ ]:
consolidated = consolidated.merge(
    resolution_register,
    on="AGREEMENT_ID",
    how="left",
    suffixes=("", "_REGISTRY")
)

for column in ["RESOLUTION_NO", "RESOLUTION_DATE"]:
    registry_col = f"{column}_REGISTRY"
    mask = consolidated[registry_col].notna()
    consolidated.loc[mask, column] = consolidated.loc[mask, registry_col]

consolidated = consolidated.drop(
    columns=["RESOLUTION_NO_REGISTRY", "RESOLUTION_DATE_REGISTRY"]
)

## 10. Final validation

In [ ]:
print("Final rows:", len(consolidated))
print("Unique record IDs:", consolidated["RECORD_ID"].nunique())
print("Unique agreement IDs:", consolidated["AGREEMENT_ID"].nunique())
print("Duplicate record IDs:", consolidated["RECORD_ID"].duplicated().sum())

assert consolidated["RECORD_ID"].duplicated().sum() == 0
assert len(consolidated) == master["RECORD_ID"].nunique()

## 11. Export a public demo output

In [ ]:
output_dir = Path("demo_output")
output_dir.mkdir(exist_ok=True)

consolidated.to_excel(
    output_dir / "consolidated_demo_2023.xlsx",
    index=False
)

pd.DataFrame(traceability).to_excel(
    output_dir / "traceability_demo_2023.xlsx",
    index=False
)

print("Demo files exported.")

## Key Takeaway

The 2023 workflow demonstrates a more mature consolidation pattern:

1. build a validated master dataset
2. incorporate same-structure updates
3. apply source-specific priority rules
4. preserve traceability
5. validate the final record universe before Power BI reporting

This public notebook keeps the technical methodology while removing all confidential operational data.